In [ ]:
!pip install earthengine-api rasterio -q

import ee
ee.Authenticate()
ee.Initialize(project='ocean-modeling')

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import rasterio

START = '2020-01-01'
END   = '2023-12-31'
BUOYS = ['41013', '41025', '41110']
NC_BOX = [-77.5, 33.5, -75.5, 36.5]
nc_coast = ee.Geometry.Rectangle(NC_BOX)

buoy_coords = {
    '41013': (-77.743, 33.436),
    '41025': (-75.402, 35.006),
    '41110': (-76.949, 34.840),
}

In [ ]:
buoy_df = buoy_df.rename(columns={
    '#YY': 'year', 'MM': 'month', 'DD': 'day', 'hh': 'hour', 'mm': 'minute',
    'WDIR': 'wind_dir', 'WSPD': 'wind_speed', 'WVHT': 'wave_height',
    'PRES': 'pressure', 'ATMP': 'air_temp', 'WTMP': 'SST'
})

buoy_df['timestamp'] = pd.to_datetime(buoy_df[['year','month','day','hour','minute']])
buoy_df.replace(99.0, np.nan, inplace=True)
buoy_df.replace(999.0, np.nan, inplace=True)
buoy_df.replace(9999.0, np.nan, inplace=True)

buoy_daily = buoy_df.set_index('timestamp').groupby('buoy_id').resample('D').agg({
    'wave_height': 'mean', 'wind_speed': 'mean', 'wind_dir': 'mean',
    'SST': 'mean', 'pressure': 'mean', 'air_temp': 'mean'
}).reset_index()

print(buoy_daily.shape)
buoy_daily.head()

(4383, 8)


,buoy_id,timestamp,wave_height,wind_speed,wind_dir,SST,pressure,air_temp
0,41013,2020-01-01,1.75375,9.172917,288.444444,19.156944,1013.677083,13.688889
1,41013,2020-01-02,0.68250,3.275000,240.666667,20.378472,1018.245833,15.179167
2,41013,2020-01-03,1.42250,8.640972,205.680556,21.556944,1015.652778,22.165972
3,41013,2020-01-04,2.41250,9.529167,226.763889,19.975694,1010.542361,21.172222
4,41013,2020-01-05,2.29750,11.915278,301.493056,18.355556,1017.790972,12.147917


In [ ]:
cdl = ee.ImageCollection('USDA/NASS/CDL') \
    .filterDate(START, END) \
    .filterBounds(nc_coast) \
    .select('cropland')

cdl_task = ee.batch.Export.image.toDrive(
    image=cdl.mode(),
    description='cdl_nc_coast',
    folder='GEE_exports',
    region=nc_coast,
    scale=30,
    fileFormat='GeoTIFF'
)
cdl_task.start()
print(cdl_task.status()['state'])

READY


In [ ]:
print(cdl_task.status()['state'])

COMPLETED


In [ ]:
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
    .filterDate(START, END) \
    .filterBounds(nc_coast) \
    .select(['temperature_2m', 'total_precipitation_sum',
             'u_component_of_wind_10m', 'v_component_of_wind_10m'])

buoy_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([-76.949, 34.840]), {'buoy_id': '41110'}),
])

def sample_era5(image):
    date = image.date().format('YYYY-MM-dd')
    sampled = image.sampleRegions(collection=buoy_points, scale=11132, geometries=False)
    return sampled.map(lambda f: f.set('date', date))

era5_tasks = []
for year in range(2020, 2024):
    era5_yr = era5.filterDate(f'{year}-01-01', f'{year+1}-01-01')
    ts = era5_yr.map(sample_era5).flatten()
    task = ee.batch.Export.table.toDrive(
        collection=ts,
        description=f'era5_41110_{year}',
        folder='GEE_exports',
        fileFormat='CSV'
    )
    task.start()
    era5_tasks.append(task)
    print(f'{year}: {task.status()["state"]}')

2020: READY
2021: READY
2022: READY
2023: READY


In [ ]:
cdl_path = '/content/drive/MyDrive/GEE_exports/cdl_nc_coast.tif'
with rasterio.open(cdl_path) as src:
    cdl_data = src.read(1)
    transform = src.transform

valid_rows, valid_cols = np.where((cdl_data != 0) & (~np.isnan(cdl_data)))
lons = transform.c + valid_cols * transform.a
lats = transform.f + valid_rows * transform.e

valid_cdl_df = pd.DataFrame({
    'row': valid_rows, 'col': valid_cols,
    'lon': lons, 'lat': lats,
    'cropland_type': cdl_data[valid_rows, valid_cols]
})

rows = []
for buoy_id, (lon, lat) in buoy_coords.items():
    valid_cdl_df['dist'] = np.sqrt((valid_cdl_df['lon']-lon)**2 + (valid_cdl_df['lat']-lat)**2)
    nearest = valid_cdl_df.loc[valid_cdl_df['dist'].idxmin()]
    rows.append({'buoy_id': buoy_id, 'cropland_type': nearest['cropland_type']})

cdl_df = pd.DataFrame(rows)
print(cdl_df)

  buoy_id  cropland_type
0   41013          131.0
1   41025          111.0
2   41110          142.0


In [ ]:
era5_yearly = []
for year in range(2020, 2024):
    df = pd.read_csv(f'/content/drive/MyDrive/GEE_exports/era5_41110_{year}.csv')
    era5_yearly.append(df)

era5_ts = pd.concat(era5_yearly)
era5_ts['buoy_id'] = era5_ts['buoy_id'].astype(str)
era5_ts['date'] = pd.to_datetime(era5_ts['date'])
era5_ts = era5_ts.rename(columns={
    'total_precipitation_sum': 'total_precipitation',
    'u_component_of_wind_10m': 'u_wind',
    'v_component_of_wind_10m': 'v_wind'
})
print(era5_ts.shape)
print(era5_ts['buoy_id'].value_counts())

(1460, 8)
buoy_id
41110    1460
Name: count, dtype: int64


In [ ]:
buoy_daily['buoy_id'] = buoy_daily['buoy_id'].astype(str)
buoy_daily['timestamp'] = pd.to_datetime(buoy_daily['timestamp'])

merged = buoy_daily[buoy_daily['buoy_id'] == '41110'].reset_index(drop=True)

merged = merged.merge(
    era5_ts[['buoy_id','date','temperature_2m','total_precipitation','u_wind','v_wind']],
    left_on=['buoy_id','timestamp'], right_on=['buoy_id','date'], how='left'
).drop(columns='date')

merged = merged.merge(cdl_df[['buoy_id','cropland_type']], on='buoy_id', how='left')

print(merged.shape)
merged.head()

(1461, 13)


,buoy_id,timestamp,wave_height,wind_speed,wind_dir,SST,pressure,air_temp,temperature_2m,total_precipitation,u_wind,v_wind,cropland_type
0,41110,2020-01-01,0.751875,NaN,NaN,15.752083,NaN,NaN,282.956753,2.145767e-06,3.656345,-0.302013,142.0
1,41110,2020-01-02,0.453958,NaN,NaN,15.231250,NaN,NaN,283.567308,8.761882e-07,1.425865,1.161455,142.0
2,41110,2020-01-03,0.799362,NaN,NaN,14.840426,NaN,NaN,290.960683,3.710985e-05,2.777287,4.178292,142.0
3,41110,2020-01-04,1.183958,NaN,NaN,15.335417,NaN,NaN,292.144864,4.783556e-03,4.275726,3.913683,142.0
4,41110,2020-01-05,0.888750,NaN,NaN,15.462500,NaN,NaN,282.884910,2.384269e-04,5.230590,-2.394793,142.0


In [ ]:
merged.to_csv('/content/drive/MyDrive/GEE_exports/nc_coast_final.csv', index=False)
print('Done!')

Done!


In [ ]:
print(merged.info())
print(merged.isnull().sum())
print(merged.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   buoy_id              1461 non-null   object        
 1   timestamp            1461 non-null   datetime64[ns]
 2   wave_height          1374 non-null   float64       
 3   wind_speed           0 non-null      float64       
 4   wind_dir             0 non-null      float64       
 5   SST                  1374 non-null   float64       
 6   pressure             0 non-null      float64       
 7   air_temp             0 non-null      float64       
 8   temperature_2m       1460 non-null   float64       
 9   total_precipitation  1460 non-null   float64       
 10  u_wind               1460 non-null   float64       
 11  v_wind               1460 non-null   float64       
 12  cropland_type        1461 non-null   float64       
dtypes: datetime64[ns](1), float64(11)

In [ ]:
merged = merged.drop(columns=['wind_speed', 'wind_dir', 'pressure', 'air_temp'])
merged = merged.dropna()

print(merged.shape)
print(merged.isnull().sum())

(1373, 9)
buoy_id                0
timestamp              0
wave_height            0
SST                    0
temperature_2m         0
total_precipitation    0
u_wind                 0
v_wind                 0
cropland_type          0
dtype: int64


In [ ]:
merged.head(10)

,buoy_id,timestamp,wave_height,SST,temperature_2m,total_precipitation,u_wind,v_wind,cropland_type
0,41110,2020-01-01,0.751875,15.752083,282.956753,2.145767e-06,3.656345,-0.302013,142.0
1,41110,2020-01-02,0.453958,15.231250,283.567308,8.761882e-07,1.425865,1.161455,142.0
2,41110,2020-01-03,0.799362,14.840426,290.960683,3.710985e-05,2.777287,4.178292,142.0
3,41110,2020-01-04,1.183958,15.335417,292.144864,4.783556e-03,4.275726,3.913683,142.0
4,41110,2020-01-05,0.888750,15.462500,282.884910,2.384269e-04,5.230590,-2.394793,142.0
5,41110,2020-01-06,0.473750,15.197917,282.458651,8.523463e-07,2.584698,0.333988,142.0
6,41110,2020-01-07,0.554167,14.606250,284.269163,2.342761e-04,1.674250,1.391481,142.0
7,41110,2020-01-08,0.566596,14.093617,282.307717,0.000000e+00,3.723203,-0.232716,142.0
8,41110,2020-01-09,0.763191,13.778723,281.547414,4.410744e-07,-1.378433,-2.307266,142.0
9,41110,2020-01-10,1.151042,14.043750,287.805613,0.000000e+00,-2.185127,1.733868,142.0


In [ ]:
merged = merged.drop(columns=['cropland_type'])
print(merged.shape)
merged.to_csv('/content/drive/MyDrive/GEE_exports/nc_coast_final.csv', index=False)
print(merged.shape)
merged.head()

(1373, 8)


,buoy_id,timestamp,wave_height,SST,temperature_2m,total_precipitation,u_wind,v_wind
0,41110,2020-01-01,0.751875,15.752083,282.956753,2.145767e-06,3.656345,-0.302013
1,41110,2020-01-02,0.453958,15.231250,283.567308,8.761882e-07,1.425865,1.161455
2,41110,2020-01-03,0.799362,14.840426,290.960683,3.710985e-05,2.777287,4.178292
3,41110,2020-01-04,1.183958,15.335417,292.144864,4.783556e-03,4.275726,3.913683
4,41110,2020-01-05,0.888750,15.462500,282.884910,2.384269e-04,5.230590,-2.394793
